# CauNagi downstream analysis tutorial

This notebook uses completed CauNagi output files directly from `temp_path/`. It does not retrain CauNagi and does not regenerate marker files. The existing downstream scripts read the CauNagi outputs, calculate downstream scores, and save their new tables under `results/`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), Path.cwd().parent) if (path / "Main_code").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not locate the repository root containing Main_code/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Adjust this only if your CauNagi output directory is elsewhere.
TEMP_PATH = next(
    (
        path
        for path in (PROJECT_ROOT / "temp_path", PROJECT_ROOT.parent / "temp_path")
        if path.is_dir()
    ),
    PROJECT_ROOT / "temp_path",
)

FINAL_ITERATION = 0  # Set this to the completed iteration number.
ITERATION_DIR = TEMP_PATH / str(FINAL_ITERATION)
STAGEDATA_DIR = ITERATION_DIR / "stagedata"
IDREM_RESULTS_DIR = ITERATION_DIR / "idremResults"
EDGES_PATH = ITERATION_DIR / "edges.txt"

# New downstream files are saved separately from the original CauNagi outputs.
RESULTS_ROOT = PROJECT_ROOT / "results" / "downstream_tutorial"
DRIVER_RESULTS_DIR = RESULTS_ROOT / "driver_genes"
CASCADE_RESULTS_DIR = RESULTS_ROOT / "cascade_classification"

print(f"CauNagi output directory: {TEMP_PATH}")
print(f"Selected iteration: {FINAL_ITERATION}")

## 1. Check the CauNagi files used by downstream analysis

The downstream scripts read these files from `temp_path`:

- `<iteration>/stagedata/*.h5ad`: staged expression data and `geneWeight`;
- `<iteration>/idremResults/`: iDREM trajectories and TF evidence;
- `<iteration>/edges.txt`: temporal graph;
- `hcmarkers.pkl`: hierarchical-clustering markers;
- `dynamic_markers.pkl`: dynamic progression markers;
- optional `NicheNet_human.csv`: prior regulatory network.

In [ ]:
required_inputs = [
    STAGEDATA_DIR / "0.h5ad",
    STAGEDATA_DIR / "dataset.h5ad",
    STAGEDATA_DIR / "attribute.pkl",
    IDREM_RESULTS_DIR,
    EDGES_PATH,
    TEMP_PATH / "hcmarkers.pkl",
    TEMP_PATH / "dynamic_markers.pkl",
]

for path in required_inputs:
    print(f"{'OK' if path.exists() else 'MISSING'}  {path}")

missing_inputs = [path for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(
        "Some required CauNagi outputs are missing. "
        "Complete the CauNagi iteration and marker analysis first."
    )

## 2. Run the existing five-dimensional driver-gene analysis

This directly calls `Candidate_Regulator_Screening/driver_gene_identification.py`. The script itself obtains the five dimensions from `temp_path`; this tutorial does not manually extract or recompute them.

In [ ]:
from Candidate_Regulator_Screening.driver_gene_identification import (
    main as run_driver_gene_analysis,
)

RUN_DRIVER_ANALYSIS = False
PRIOR_NETWORK_PATH = TEMP_PATH / "NicheNet_human.csv"

if RUN_DRIVER_ANALYSIS:
    DRIVER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    driver_gene_table = run_driver_gene_analysis(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        prior_net_path=(
            str(PRIOR_NETWORK_PATH)
            if PRIOR_NETWORK_PATH.is_file()
            else None
        ),
        output_dir=str(DRIVER_RESULTS_DIR),
        species="human",
    )
    display(driver_gene_table.head(30))

## 3. Run the existing cell-type-specific driver-gene analysis

This directly calls `Candidate_Regulator_Screening/per_celltype_driver_genes.py`. It reads staged data and iDREM results from `temp_path`, then writes cell-type driver-gene tables to `results/downstream_tutorial/driver_genes/`.

In [ ]:
from Candidate_Regulator_Screening import per_celltype_driver_genes

RUN_CELLTYPE_DRIVER_ANALYSIS = False

if RUN_CELLTYPE_DRIVER_ANALYSIS:
    DRIVER_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    celltype_driver_results = per_celltype_driver_genes.main(
        temp_path=str(TEMP_PATH),
        final_iteration=FINAL_ITERATION,
        output_dir=str(DRIVER_RESULTS_DIR),
    )

## 4. Run the existing cascade-classification analysis

The cascade script uses the cell-type driver-gene tables produced in the previous step. It does not read raw expression data directly; those driver-gene tables were calculated from the CauNagi outputs in `temp_path`.

In [ ]:
import cascade_classification.cascade_classification as cascade

RUN_CASCADE_ANALYSIS = False

if RUN_CASCADE_ANALYSIS:
    required_tables = [
        DRIVER_RESULTS_DIR / f"{cell_type}_driver_genes.csv"
        for cell_type in ["HSPC", "GMP", "Monocyte", "Neutrophil"]
    ]
    missing_tables = [path for path in required_tables if not path.is_file()]
    if missing_tables:
        raise FileNotFoundError(f"Missing driver-gene tables: {missing_tables}")

    CASCADE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    cascade.BASE_DIR = str(DRIVER_RESULTS_DIR)
    cascade.OUT_DIR = str(CASCADE_RESULTS_DIR)
    cascade.main()

## 5. Obtain cascade genes

The cascade script writes `gene_cascade_classification.csv`. The following code extracts both all classified cascade genes and the stricter cascade-core genes.

In [ ]:
classification_path = CASCADE_RESULTS_DIR / "gene_cascade_classification.csv"

if not classification_path.is_file():
    raise FileNotFoundError(
        f"Run the cascade-classification step first: {classification_path}"
    )

classification = pd.read_csv(classification_path)

cascade_categories = [
    "Cascade_Core_Uniform",
    "Cascade_Core_Decay",
    "HSC_Initiator",
    "GMP_Propagator",
    "Mono_Amplifier",
    "Neut_Amplifier",
]
core_categories = [
    "Cascade_Core_Uniform",
    "Cascade_Core_Decay",
]

cascade_genes = (
    classification[
        classification["cascade_category"].isin(cascade_categories)
    ]
    .sort_values(["cascade_category", "HPS", "CTS"], ascending=[True, False, False])
    .reset_index(drop=True)
)

cascade_core_genes = (
    classification[
        classification["cascade_category"].isin(core_categories)
    ]
    .sort_values(["HPS", "CTS"], ascending=[False, False])
    .reset_index(drop=True)
)

cascade_genes.to_csv(
    CASCADE_RESULTS_DIR / "cascade_genes.csv",
    index=False,
    encoding="utf-8-sig",
)
cascade_core_genes.to_csv(
    CASCADE_RESULTS_DIR / "cascade_core_genes.csv",
    index=False,
    encoding="utf-8-sig",
)

print(f"All cascade genes: {len(cascade_genes)}")
print(f"Cascade-core genes: {len(cascade_core_genes)}")
display(cascade_core_genes.head(30))

## 6. Review downstream results

In [ ]:
result_files = sorted(
    path
    for result_dir in [DRIVER_RESULTS_DIR, CASCADE_RESULTS_DIR]
    if result_dir.is_dir()
    for path in result_dir.iterdir()
    if path.is_file()
)

display(
    pd.DataFrame(
        {
            "file": [str(path.relative_to(PROJECT_ROOT)) for path in result_files],
            "size_mb": [
                round(path.stat().st_size / 1024**2, 3)
                for path in result_files
            ],
        }
    )
)